# Model Training and Evaluation Setup

This setup defines which data to use for training and testing a model.

There are two standard training and test options.
1. train_specification = ‘1_to_25_80p’ with test_specification = '26_to_50_100p'
2. train_specification = ‘26_to_50_80p’ with test_specification = '1_to_25_100p'

e.g. 1_to_25_80p: _1_to_25 means question block 1, with 80 percent of the data_

In [1]:
train_specification = '1_to_25_80p'
test_specification = '26_to_50_100p'

___
--There is a separate notebook for the traceability of the training- and testdata: \_split_train_test_data.ipynb--

# Dataset Preperation

In [2]:
import pandas as pd
train_df = pd.read_json(f'training_splits/train_{train_specification}.json')
train_df.columns = ['query', 'answer', 'label']

In [3]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "sentence1": [row['query'] for _, row in train_df.iterrows()],
    "sentence2": [row['answer'] for _, row in train_df.iterrows()],
    "label": [int(row['label']) for _, row in train_df.iterrows()],
})

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Training

In [4]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses, SentenceTransformerTrainingArguments

In [5]:
basemodel = SentenceTransformer('all-mpnet-base-v2')
model_name = 'new_trained_sentence_transformer'  # arbitrary

In [6]:
args = SentenceTransformerTrainingArguments(model_name,  num_train_epochs = 1)

In [7]:
trainer = SentenceTransformerTrainer(
    args=args,
    model=basemodel,
    train_dataset=train_dataset,    
    loss=losses.ContrastiveLoss(basemodel)
)
trainer.train()

Step,Training Loss


TrainOutput(global_step=37, training_loss=0.02253821733835581, metrics={'train_runtime': 99.8232, 'train_samples_per_second': 2.925, 'train_steps_per_second': 0.371, 'total_flos': 0.0, 'train_loss': 0.02253821733835581, 'epoch': 1.0})

In [8]:
trainer.save_model(model_name)